In [17]:
import matplotlib.pyplot as plt
import datetime as dt
import pandas as pd
import numpy as np
import scipy as sp
import json
import subprocess

from pathlib import Path
import soundfile as sf

In [18]:
import folium
from folium.plugins import MeasureControl

In [19]:
map = folium.Map(location=[47.6543389, -122.2949448], zoom_start=24, control_scale=True, max_zoom=28)
carp_pond_bench_v1 = [47.6543389, -122.2949448]
carp_pond_bench_v2 = [47.6543426, -122.2949264]

am_iphone_gps_right = [47.654346, -122.294944]
am_iphone_gps_left = [47.654350, -122.294943]

instru1_gps_right = [47.65434, -122.29496]
instru1_gps_left = [47.65433, -122.29492]

photo_dir = Path.home() / "Downloads/audiomoth50_fencepost_pics"
photo_paths = sorted(photo_dir.rglob("*.HEIC"))

photo_metadata = []
if len(photo_paths) > 0:
    photo_metadata = json.loads(subprocess.check_output(["exiftool", "-json", "-n", *[str(path) for path in photo_paths]], text=True))

photo_marker_locations = []
for photo_metadata_row in photo_metadata:
    if "GPSLatitude" in photo_metadata_row and "GPSLongitude" in photo_metadata_row:
        photo_name = Path(photo_metadata_row["SourceFile"]).parent.name
        photo_location = [photo_metadata_row["GPSLatitude"], photo_metadata_row["GPSLongitude"]]
        photo_marker_locations.append((photo_name, photo_location, "lime"))

measurement_csv_path = photo_dir / "post_A7/Raw Data.csv"
measurement_marker_locations = []
if measurement_csv_path.exists():
    measurement_points = pd.read_csv(measurement_csv_path)
    measurement_points = measurement_points[int(len(measurement_points) * (0.4)):int(len(measurement_points) * (0.6))]
    measurement_points = measurement_points[["Latitude (°)", "Longitude (°)"]].dropna().copy()
    for measurement_row_i, measurement_row in measurement_points.iterrows():
        measurement_name = measurement_csv_path.parent.name
        measurement_location = [measurement_row["Latitude (°)"], measurement_row["Longitude (°)"]]
        measurement_marker_locations.append((measurement_name, measurement_location, "magenta"))

marker_locations = [("Google Earth Bv1", carp_pond_bench_v1, "red"),
                    ("Google Earth Bv2", carp_pond_bench_v2, "red"),
                    ("iPhone GPS point", am_iphone_gps_right, "yellow"),
                    ("iPhone GPS point", am_iphone_gps_left, "yellow"),
                    ("Instru1 GPS point", instru1_gps_right, "blue"),
                    ("Instru1 GPS point", instru1_gps_left, "blue")] + photo_marker_locations + measurement_marker_locations

for name, location, fill_color in marker_locations:
    coordinate_label = f"{name}: {location[0]:.6f}, {location[1]:.6f}"
    folium.CircleMarker(location=location, radius=3, color="black", weight=1,
                        fill=True, fill_color=fill_color, fill_opacity=1,
                        tooltip=folium.Tooltip(coordinate_label, permanent=True,
                        direction="right", offset=(8, 0))).add_to(map)

map.fit_bounds([location for _, location, _ in marker_locations])

# The ruler button measures a line drawn between any points on the map.
MeasureControl(position="topleft", primary_length_unit="meters", secondary_length_unit="feet").add_to(map)

map


   13 image files read


In [27]:
from pyproj import Transformer

post_order = ["post_A7", "post_A1", "post_A4", "post_E2", "post_E4", "post_E8", "post_E9"]

post_points = pd.DataFrame([{"post": name, "lat": location[0], "lon": location[1]}
                            for name, location, fill_color in marker_locations
                            if name in post_order])

missing_posts = [post_name for post_name in post_order if post_name not in set(post_points["post"])]
if len(missing_posts) > 0:
    raise ValueError(f"Missing post points for: {missing_posts}")

average_post_locations = post_points.groupby("post")[["lat", "lon"]].mean().loc[post_order]
average_post_locations

,lat,lon
post,,
post_A7,47.655097,-122.296698
post_A1,47.655707,-122.296649
post_A4,47.655381,-122.296653
post_E2,47.654839,-122.295539
post_E4,47.654944,-122.294883
post_E8,47.654442,-122.294875
post_E9,47.654286,-122.295294


In [21]:
# WGS84 lat/lon/elevation -> Earth-centered Earth-fixed XYZ meters
lla_to_ecef = Transformer.from_crs(
    "EPSG:4979",  # lat, lon, height
    "EPSG:4978",  # geocentric X, Y, Z
    always_xy=True,
)

def distance_3d_m(lat1, lon1, elev1, lat2, lon2, elev2):
    x1, y1, z1 = lla_to_ecef.transform(lon1, lat1, elev1)
    x2, y2, z2 = lla_to_ecef.transform(lon2, lat2, elev2)

    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

In [22]:
average_post_distance_m = {}
for post_name in post_order:
    average_post_distance_m[post_name] = []
    for other_post_name in post_order:
        if other_post_name == post_name:
            continue

        lat1 = average_post_locations.loc[post_name, "lat"]
        lon1 = average_post_locations.loc[post_name, "lon"]
        lat2 = average_post_locations.loc[other_post_name, "lat"]
        lon2 = average_post_locations.loc[other_post_name, "lon"]
        distance_m = distance_3d_m(lat1, lon1, 0, lat2, lon2, 0)
        distance_name = f"average_{post_name.replace('_', '')}_to_average_{other_post_name.replace('_', '')}"
        average_post_distance_m[post_name].append((distance_name, distance_m))

In [26]:
average_post_distance_m

{'post_A7': [('average_postA7_to_average_postA1', 67.93873769407455),
  ('average_postA7_to_average_postA4', 31.681713039009768),
  ('average_postA7_to_average_postE2', 91.69696742863886),
  ('average_postA7_to_average_postE4', 137.38210580687522),
  ('average_postA7_to_average_postE8', 155.14347675152365),
  ('average_postA7_to_average_postE9', 138.7520323305727)],
 'post_A1': [('average_postA1_to_average_postA7', 67.93873769407455),
  ('average_postA1_to_average_postA4', 36.34163661301001),
  ('average_postA1_to_average_postE2', 127.59347415212547),
  ('average_postA1_to_average_postE4', 157.44991229724167),
  ('average_postA1_to_average_postE8', 193.81945616372317),
  ('average_postA1_to_average_postE9', 187.9556612746589)],
 'post_A4': [('average_postA4_to_average_postA7', 31.681713039009768),
  ('average_postA4_to_average_postA1', 36.34163661301001),
  ('average_postA4_to_average_postE2', 103.09608926418578),
  ('average_postA4_to_average_postE4', 141.4907867498218),
  ('average_p